# AbstractTensor — Manifold, tangent bundle and frame bundle

This notebook explains the structures defined with @def_manifold macro and their hierarchy 

---
## 1. Loading package

In [1]:
using AbstractTensors

[ Info: Precompiling AbstractTensors [55c8b40c-cbfa-4141-803e-4831c9841971] (cache misses: include_dependency fsize change (4), wrong source (4), mismatched flags (10))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


---
## 2. Manifolds and def_manifold macro
---

### a. Manifold type and definition

#### Documentation access
- `@doc Manifold` — shows the docstring directly (julia macro)
- `?Manifold` — same, with search results prepended (Jupyter help mode)

In [2]:
@doc Manifold

```julia
Manifold
```

Struct representing a registered differentiable manifold. Instances are created by [`@def_manifold`](@ref) and bound to a variable in the caller's scope.

Provides dot access to all metadata:

```julia
M.name              # :M
M.dim               # 4
M.tangent_bundle    # :tangentM
M.cotangent_bundle  # :cotangentM
M.vbundles          # [:tangentM, :cotangentM]
```

### Fields

  * `name`             : manifold name, e.g. `:M`
  * `dim`              : dimension
  * `tangent_bundle`   : name of the tangent bundle, e.g. `:tangentM`
  * `cotangent_bundle` : name of the cotangent (dual) bundle, e.g. `:cotangentM`
  * `vbundles`         : names of vector bundles with manifold as base manifold


In [3]:
?@def_manifold

```julia
@def_manifold name dim coord_indices basis_indices [kwargs...]
```

Define a new manifold and automatically create its tangent and cotangent bundles, coordinate frames, and moving frame bundles.

Both index lists are **required**. Each list should have at least 4 symbols (a warning is issued if fewer).

Bind in the caller's scope:

  * `name`            → a [`Manifold`](@ref) instance
  * `tangent<name>`   → a [`VBundle`](@ref) instance (`isdual = false`)
  * `cotangent<name>` → a [`VBundle`](@ref) instance (`isdual = true`)
  * `frame<name>`, `coframe<name>`, moving basis symbols (default `e`, `θ`)

Coordinate indices (first list) → [`CoordinateIndex`](@ref):

```julia
a1          # CoordinateIndex(:a1, :tangentM)   — contravariant
-a1         # CoordinateIndex(:a1, :cotangentM) — covariant
```

Basis indices (second list) → [`BasisIndex`](@ref):

```julia
A1          # BasisIndex(:A1, :tangentM)   — contravariant
-A1         # BasisIndex(:A1, :cotangentM) — covariant
```

#### Examples

```julia
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]
@def_manifold M d [b1, b2, b3, b4] [B1, B2, B3, B4]   # parametric dimension
```


In [17]:
@def_manifold M 4 [a1, a2, a3, a4] [A1, A2, A3, A4]

Defined manifold M of dimension 4
Defined vector bundle tangentM with coordinate basis ∂ and its dual cotangentM with coordinate basis dx
Defined frame bundle frameM (moving frame e) and coframe bundle coframeM (moving coframe θ) over M


In [18]:
_MANIFOLDS

Dict{Symbol, Manifold} with 1 entry:
  :M => Manifold(M, dim=4, TBundle=tangentM, CBundle=cotangentM)

### b. Tangent and cotangent bundles 

In [19]:
tangentM

VBundle(tangentM, tangent, dual=cotangentM, manifold=M, dim=4, bases=[Basis(∂, tangentM, coord), Basis(e, tangentM, moving)])

In [33]:
display(tangentM.bases)
display(tangentM.indices)
display(tangentM.basis_indices)

2-element Vector{Basis}:
 Basis(∂, tangentM, coord)
 Basis(e, tangentM, moving)

4-element Vector{AbstractIndex}:
 +a1 ∈ tangentM (coord)
 +a2 ∈ tangentM (coord)
 +a3 ∈ tangentM (coord)
 +a4 ∈ tangentM (coord)

4-element Vector{BasisIndex}:
 +A1 ∈ tangentM (basis)
 +A2 ∈ tangentM (basis)
 +A3 ∈ tangentM (basis)
 +A4 ∈ tangentM (basis)

In [20]:
cotangentM

VBundle(cotangentM, cotangent, dual=tangentM, manifold=M, dim=4, bases=[Basis(dx, cotangentM, coord), Basis(θ, cotangentM, moving)])

In [34]:
display(cotangentM.bases)
display(cotangentM.indices)
display(cotangentM.basis_indices)

2-element Vector{Basis}:
 Basis(dx, cotangentM, coord)
 Basis(θ, cotangentM, moving)

4-element Vector{AbstractIndex}:
 -a1 ∈ cotangentM (coord)
 -a2 ∈ cotangentM (coord)
 -a3 ∈ cotangentM (coord)
 -a4 ∈ cotangentM (coord)

BasisIndex[]

In [35]:
dx

Basis(dx, cotangentM, coord)

In [38]:
dx[a1]

dx[a1]

In [39]:
dx[-a1]

LoadError: BasisElement: index vbundle :cotangentM is not the dual of basis vbundle :cotangentM. Expected index from :tangentM. Use dx[...] with an index from the dual bundle.

In [25]:
∂

Basis(∂, tangentM, coord)

In [42]:
∂[-a1]

∂[-a1]

In [36]:
e

Basis(e, tangentM, moving)

In [44]:
e[-A1]

e[-A1]

In [46]:
e[A1]

LoadError: BasisElement: index vbundle :tangentM is not the dual of basis vbundle :tangentM. Expected index from :cotangentM. Use e[...] with an index from the dual bundle.

In [47]:
e[a1]

LoadError: BasisElement: index vbundle :tangentM is not the dual of basis vbundle :tangentM. Expected index from :cotangentM. Use e[...] with an index from the dual bundle.

In [48]:
e[-a1]//separateM

e[-a1]

In [50]:
θ[A1]

θ[A1]

In [51]:
θ[-A1]

LoadError: BasisElement: index vbundle :cotangentM is not the dual of basis vbundle :cotangentM. Expected index from :tangentM. Use θ[...] with an index from the dual bundle.

In [52]:
θ[a1]

θ[a1]

### c. Frame bundle 

In [26]:
frameM

VBundle,tangentM
Dual,coframeM
Moving basis,e


In [28]:
coframeM

VBundle,cotangentM
Dual,frameM
Moving basis,θ
